In [4]:
import os
import sys

# Manually set environment variables BEFORE importing findspark or pyspark
os.environ['JAVA_HOME'] = '/usr/local/java'
os.environ['SPARK_HOME'] = '/usr/local/spark'
os.environ['HADOOP_CONF_DIR'] = '/usr/local/hadoop/etc/hadoop'
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

import findspark
findspark.init()

import pyspark
from pyspark.sql import SparkSession
import matplotlib.pyplot as plt
import pandas as pd

print(f"Spark Home: {os.environ.get('SPARK_HOME')}")
print(f"Java Home: {os.environ.get('JAVA_HOME')}")
print(f"Spark Version: {pyspark.__version__}")

Spark Home: /usr/local/spark
Java Home: /usr/local/java
Spark Version: 3.5.1


# Business Analysis: Profitability by Product Category
In this notebook, we use **Spark SQL** to query the Hive database to identify which product categories are the most profitable. This insight helps the business decide which product lines to expand or optimize.

### Tools Used:
- **PySpark**: For distributed data processing and Spark SQL.
- **Hive**: As the data warehouse storing the Superstore data.
- **Matplotlib**: For visualizing the analysis results.

In [8]:
from pyspark.sql import SparkSession

# Stop any existing context to avoid conflicts
try:
    if 'spark' in locals() or 'spark' in globals():
        spark.stop()
except:
    pass

# To connect to Apache Hive 3.1.3, we use the matching metastore version 3.1.2.
# We must use the direct IP to bypass the 'Illegal character in hostname' error
# caused by underscores in the Docker network name during Java URI parsing.

# spark = SparkSession.builder \
#     .appName("Superstore Profit Analysis") \
#     .master("spark://namenode:7077") \
#     .config("spark.sql.warehouse.dir", "/user/hive/warehouse") \
#     .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
#     .config("spark.sql.hive.metastore.version", "3.1.2") \
#     .config("spark.sql.hive.metastore.jars", "maven") \
#     .enableHiveSupport() \
#     .getOrCreate()

spark = SparkSession.builder \
    .appName("Superstore Profit Analysis") \
    .master("spark://namenode:7077") \
    .config("spark.sql.warehouse.dir", "/opt/hive/data/warehouse") \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.hive.metastore.version", "2.3.9") \
    .config("spark.sql.hive.metastore.jars", "builtin") \
    .enableHiveSupport() \
    .getOrCreate()

# Verify connection and show tables
try:
    print("Listing tables in Hive...")
    spark.sql("SHOW TABLES").show()
    spark.sql("SELECT * FROM customers;").show()
except Exception as e:
    print(f"Error querying Hive: {e}")

Listing tables in Hive...
+---------+-----------+-----------+
|namespace|  tableName|isTemporary|
+---------+-----------+-----------+
|  default|  customers|      false|
|  default|  locations|      false|
|  default|order_items|      false|
|  default|     orders|      false|
|  default|   products|      false|
+---------+-----------+-----------+

Error querying Hive: An error occurred while calling o215.showString.
: org.apache.hadoop.mapred.InvalidInputException: Input path does not exist: file:/opt/hive/data/warehouse/customers
	at org.apache.hadoop.mapred.FileInputFormat.singleThreadedListStatus(FileInputFormat.java:304)
	at org.apache.hadoop.mapred.FileInputFormat.listStatus(FileInputFormat.java:244)
	at org.apache.hadoop.mapred.FileInputFormat.getSplits(FileInputFormat.java:332)
	at org.apache.spark.rdd.HadoopRDD.getPartitions(HadoopRDD.scala:208)
	at org.apache.spark.rdd.RDD.$anonfun$partitions$2(RDD.scala:294)
	at scala.Option.getOrElse(Option.scala:189)
	at org.apache.spark.r

26/04/21 20:13:42 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.


## 1. Extract Insights using Spark SQL
We will join `order_items` and `products` tables to calculate total sales and total profit for each product category.

In [3]:
# Spark SQL Query
query = """
SELECT 
    p.category, 
    SUM(oi.sales) as total_sales, 
    SUM(oi.profit) as total_profit
FROM 
    order_items oi
JOIN 
    products p ON oi.product_id = p.product_id
GROUP BY 
    p.category
ORDER BY 
    total_profit DESC
"""

# Execute query and convert to Pandas for visualization
df_profit = spark.sql(query).toPandas()
print(df_profit)

Py4JJavaError: An error occurred while calling o41.collectToPython.
: org.apache.hadoop.mapred.InvalidInputException: Input path does not exist: file:/opt/hive/data/warehouse/products
	at org.apache.hadoop.mapred.FileInputFormat.singleThreadedListStatus(FileInputFormat.java:304)
	at org.apache.hadoop.mapred.FileInputFormat.listStatus(FileInputFormat.java:244)
	at org.apache.hadoop.mapred.FileInputFormat.getSplits(FileInputFormat.java:332)
	at org.apache.spark.rdd.HadoopRDD.getPartitions(HadoopRDD.scala:208)
	at org.apache.spark.rdd.RDD.$anonfun$partitions$2(RDD.scala:294)
	at scala.Option.getOrElse(Option.scala:189)
	at org.apache.spark.rdd.RDD.partitions(RDD.scala:290)
	at org.apache.spark.rdd.MapPartitionsRDD.getPartitions(MapPartitionsRDD.scala:49)
	at org.apache.spark.rdd.RDD.$anonfun$partitions$2(RDD.scala:294)
	at scala.Option.getOrElse(Option.scala:189)
	at org.apache.spark.rdd.RDD.partitions(RDD.scala:290)
	at org.apache.spark.rdd.MapPartitionsRDD.getPartitions(MapPartitionsRDD

## 2. Visualize Insights with Matplotlib
A bar chart comparing Profit and Sales across different categories.

In [ ]:
# Plotting the data
plt.figure(figsize=(10, 6))

# Profit Bar Chart
plt.bar(df_profit['category'], df_profit['total_profit'], color='skyblue', label='Total Profit')

# Adding labels and title
plt.xlabel('Category')
plt.ylabel('Amount ($)')
plt.title('Total Profit by Product Category')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.show()

## 3. Top 5 Sub-Categories Analysis
Let's drill down into the sub-categories for more granular insight.

In [ ]:
sub_category_query = """
SELECT 
    p.sub_category, 
    SUM(oi.profit) as total_profit
FROM 
    order_items oi
JOIN 
    products p ON oi.product_id = p.product_id
GROUP BY 
    p.sub_category
ORDER BY 
    total_profit DESC
LIMIT 5
"""

df_sub_profit = spark.sql(sub_category_query).toPandas()

# Visualization
plt.figure(figsize=(12, 6))
plt.barh(df_sub_profit['sub_category'], df_sub_profit['total_profit'], color='salmon')
plt.xlabel('Total Profit ($)')
plt.ylabel('Sub-Category')
plt.title('Top 5 Most Profitable Sub-Categories')
plt.gca().invert_yaxis()
plt.show()